## Recurring Donations

### Overview

Analyse a single recurring donor transaction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet('full_data.parquet', engine="pyarrow")

df.info()

In [ ]:
amount_df = pd.DataFrame(df["amount"], columns=["amount"])
amount_df.info()
print(f"minimum:{amount_df.min()} maximum:{df.amount.max()} average:{df.amount.mean()}")
ax = amount_df.plot.hist(bins=80)


In [ ]:
df.groupby("customer_name").value_counts()

Filtering certain `customer_id`.

`sort_values` by `created_utc` datetime value

View the top and bottom of the `df`

In [ ]:
#df=df.loc[df["customer_id"].notnull()]

#df.to_csv("recurring_donors.csv")

#unique_customers=set(df["customer_id"].tolist())

#print(len(unique_customers))

#fig, ax = plt.subplots(figsize=(10,10), facecolor="grey", layout="tight")
#df.plot(x="created_utc", y="amount", kind="scatter", ax=ax, c="red", marker="x", markersize=0.5)


#for customer in unique_customers:
    #customer_df = df.loc[df["customer_id"] == customer]
 
    #df.sort_values(by="created_utc", inplace=True, ascending=True)
    
    


Visualising stopped/cancelled donations

In [ ]:

#fig, ax = plt.subplots(figsize=(10,10), facecolor="grey", layout="tight")

#unique_customers=set(df["customer_id"].tolist())

#print(len(unique_customers))

#df.plot(x="created_utc", y="amount", kind="scatter", ax=ax, c="red")
 

### Value counts of the customer name and sorting out the number of donations made per customer.

In [ ]:
ranked_donors_df = df["customer_name"].value_counts().reset_index().sort_values(by="count", ascending=False)
ranked_donors_df.head(10)

**Gross lifetime donations to date**

In [ ]:
gross_volume_df = df.groupby(["customer_name"])["amount"].sum().reset_index()
gross_volume_df = gross_volume_df.sort_values(by= "amount", ascending=False)
gross_volume_df.head(10)

**Gross lifetime fees to date**

In [ ]:
gross_fee_df = df.groupby(["customer_name"])["fee"].sum().reset_index()
gross_fee_df = gross_fee_df.sort_values(by= "fee", ascending=False)
gross_fee_df.head(10)

**Example 1: of a print statement with an f string**

In [ ]:
print(f"minimum:{gross_fee_df.fee.min()} maximum:{gross_fee_df.fee.max()} average:{gross_fee_df.fee.mean()}")
#ax = amount_df.plot.hist(bins=80)

**Example 2: print statement without an f string == two different method**

In [ ]:
print("minimum:{} maximum:{} average:{}".format(gross_fee_df["fee"].min(), gross_fee_df["fee"].max(), gross_fee_df["fee"].mean()))
#ax = amount_df.plot.hist(bins=80)

**Simple Histogram of fees**

In [ ]:
ax = gross_fee_df["fee"].plot.hist(bins=40, color="skyblue", edgecolor= "black")
ax.set_xlabel("fee")
ax.set_ylabel("frequency")
ax.grid(axis="y", linestyle="--", alpha=0.7)


***Simple histogram of fees which analyses the proportion of fees relative to cost***

**Top Customers Bar Plot**

In [ ]:
ax = gross_fee_df.head(10).plot.barh(
    x="customer_name",
    y="fee",
    legend=False,
    title="Top 10 customers by Gross Fees"
)
ax.invert_yaxis()

**Seasonal Giving Trends**

In [ ]:
#Seasonal/monthly giving trends over the years
df['month_name'] = df['created_utc'].dt.strftime('%B')

months_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
monthly_totals = df.groupby('month_name')['amount'].sum().reindex(months_order)

ax = monthly_totals.plot.bar(color='Green', rot=45)
ax.set_title("Total Donations received by Month")
ax.set_ylabel("Total Funds Raised")

**Running Total of Funds**

In [ ]:
#Cumulative funds raised over the years (a running total)
df['created_utc'] = pd.to_datetime(df['created_utc'])
df_sorted = df.sort_values('created_utc')
df_sorted['cumulative_amount'] = df_sorted['amount'].cumsum()

ax = df_sorted.plot.line(x='created_utc', y='cumulative_amount', color= 'cornflowerblue', legend=False)
ax.set_title("Cumulative Donations Over Time")
ax.set_xlabel("Time")
ax.set_ylabel("Total Funds Raised ($)")

**Recurring versus one-time giver breakdown**

In [ ]:
#Recurring versus one-time giver breakdown over the years
donor_frequency = df.dropna(subset=["customer_id"])["customer_id"].value_counts()

one_time_count = (donor_frequency == 1).sum()
recurring_count = (donor_frequency > 1).sum()

fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    [one_time_count, recurring_count],
    labels=["One-Time Donors", "Recurring Donors"],
    autopct="%1.1f%%",
    startangle=140,
    colors=["#2D909B", "#2D5279"],
    pctdistance=0.8,
    textprops=dict(color="#2515B9", weight="bold"),
    wedgeprops=dict(width=0.4, edgecolor="w", linewidth=2)
)

plt.setp(autotexts, size=10, weight="bold")
ax.set_title("Donor Retention Base Profile", fontsize=13, weight="bold", pad=15)
plt.tight_layout()
plt.show()